In [14]:
import optuna
import numpy as np
import gymnasium as gym
from functools import partial
import torch.nn as nn

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor, DummyVecEnv
from industrial_inventory_env import IndustrialInventoryEnv, generate_student_config

ROLL_NUMBER = "DA25M622"
student_config = generate_student_config(ROLL_NUMBER)
print(f"Student Config for {ROLL_NUMBER}: {student_config}")

Student Config for DA25M622: {'project_version': 'IITM-6002W-RL-Inventory-2026-v1', 'roll_number': 'DA25M622', 'variant_id': 'V022', 'demand_multiplier_profile': [1.0, 1.1, 0.9], 'initial_inventory_profile': [110, 100, 90], 'lead_time_delay_profile': [0.1, 0.0, 0.05], 'declared_ranges': {'demand_multiplier': [0.85, 1.15], 'initial_inventory': [80, 120], 'lead_time_delay_probability': [0.0, 0.1]}, 'config_fingerprint': '7d77fc79debf206a'}


/home/sohang/Projects/iit-madras-web-mtech-ai/trimester3/DA6002W_Online&ReinforcementLearning/ReinforcementLearning/RL_ProjectCompetition/RL_Student_Package_2026/.venv-sb3-torch213/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def evaluate_cost(model, eval_seed=42, n_eval_episodes=15):
    """Evaluates the model and returns the average episode total cost (lower is better)."""
    eval_env = IndustrialInventoryEnv(
        student_config=student_config, 
        scenario_mode="random", 
        domain_randomization=True
    )
    episode_costs = []
    
    for ep in range(n_eval_episodes):
        obs, info = eval_env.reset(seed=eval_seed + ep)
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
        episode_costs.append(info["costs"]["episode_total"])
        
    eval_env.close()
    return float(np.mean(episode_costs))

def sample_ppo_params(trial: optuna.Trial) -> dict:
    gamma = trial.suggest_categorical("gamma", [0.98, 0.99, 0.995])
    gae_lambda = trial.suggest_categorical("gae_lambda", [0.90, 0.95, 0.98])
    learning_rate = trial.suggest_float("learning_rate", 3e-5, 8e-4, log=True)
    ent_coef = trial.suggest_float("ent_coef", 1e-5, 1e-2, log=True)
    clip_range = trial.suggest_categorical("clip_range", [0.1, 0.2, 0.3])
    n_epochs = trial.suggest_categorical("n_epochs", [5, 10])
    
    # n_steps per env (4 parallel envs total)
    n_steps = trial.suggest_categorical("n_steps", [128, 256, 512])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    
    # Ensure mini-batch size divides total rollout buffer
    while (n_steps * 4) % batch_size != 0:
        batch_size //= 2
        
    net_arch = trial.suggest_categorical("net_arch", ["small", "medium"])
    layers = [128, 128] if net_arch == "small" else [256, 256]
    
    policy_kwargs = dict(
        net_arch=dict(pi=layers, vf=layers),
        activation_fn=nn.ReLU
    )
    
    return {
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "clip_range": clip_range,
        "n_epochs": n_epochs,
        "n_steps": n_steps,
        "batch_size": batch_size,
        "policy_kwargs": policy_kwargs,
    }

def objective(trial: optuna.Trial) -> float:
    params = sample_ppo_params(trial)
    
    env_fn = lambda: IndustrialInventoryEnv(
        student_config=student_config, 
        scenario_mode="random", 
        domain_randomization=True
    )
    
    vec_env = make_vec_env(env_id=env_fn, n_envs=4, seed=2026, vec_env_cls=DummyVecEnv)
    vec_env = VecMonitor(vec_env)
    
    model = PPO(
        policy="MultiInputPolicy",
        env=vec_env,
        verbose=0,
        seed=2026,
        **params
    )
    
    # Fast trial run for candidate screening
    try:
        model.learn(total_timesteps=60_000)
        mean_cost = evaluate_cost(model, eval_seed=101, n_eval_episodes=10)
    except Exception as e:
        vec_env.close()
        raise e
        
    vec_env.close()
    return mean_cost

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=2026),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

# 20 to 30 trials are typically sufficient to beat default configurations
study.optimize(objective, n_trials=25, timeout=3600)

print("Best Trial:")
print(f"  Mean Evaluated Cost: {study.best_trial.value:,.2f}")
print("  Params: ")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

[I 2026-09-02 21:55:03,208] A new study created in memory with name: no-name-ca214836-d530-4171-b334-ba9faa3b3a72
/home/sohang/Projects/iit-madras-web-mtech-ai/trimester3/DA6002W_Online&ReinforcementLearning/ReinforcementLearning/RL_ProjectCompetition/RL_Student_Package_2026/.venv-sb3-torch213/lib/python3.12/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(
[I 2026-09-02 21:55:44,171] Trial 0 finished with value: 1641983.25 and parameters: {'gamma': 0.995, 'gae_lambda': 0.98, 'learning_rate': 5.749952436638167e-05, 'ent_coef': 0.0054343242201941545, 'clip_range': 0.2, 'n_epochs': 5, 'n_steps': 128, 'batch_size': 128, 'net_arch': 'small'}. Best is trial 0 with value: 1641983.25.
[I 2026-09-02 21:56:51,237] Trial 1 finished with value: 1417398.75 and parameters: {

In [ ]:
# need to retrain with best hyperparams as we didn't save it in optuna :(
vec_env = make_vec_env(env_id=env_fn, n_envs=4, seed=2026, vec_env_cls=DummyVecEnv)
vec_env = VecMonitor(vec_env)
best_params = study.best_trial.params
model = PPO(
    policy="MultiInputPolicy",
    env=vec_env,
    verbose=0,
    seed=2026,
    **best_params
)
model.learn(total_timesteps=60_000) 

In [ ]:
mean_cost = evaluate_cost(model, eval_seed=2026, n_eval_episodes=20)